# Notebook 04 — Unified Backtest Harness, Modeling, OOF Hybrid, CQR Quantiles & Auto-Router

**Thiết kế 1 Luồng Thống nhất (Rate-Normalized `ŷ = r̂ · active_days`, 5 Folds Tuần + Daily 365D):**
1. **Baselines chuẩn & Paper 2024:** `Seasonal_Naive (y[t−52])`, `Seasonal_Naive_Weighted ([0.4, 0.3, 0.2, 0.1])`, `Harmonic_ARIMA` (trên `qty_rate`), `Croston_SBA` (trên `qty_rate`).
2. **Mở rộng ngoại suy & giảm phương sai:** `Ridge_F1F2F3` (`StandardScaler + RidgeCV`), Ablation LightGBM (`F1`, `F1+F2`, `F1+F2+F3_Compact`, `F3_LagOnly`) kèm **TreeSHAP**, và **`LightGBM_AnchorRes_F3`** (dự báo phần dư quanh `EWMA_4` để không bị kẹt trần cây trên chuỗi tăng/giảm).
3. **Expanding-Block OOF Hybrid (`Hybrid_SARIMA_LGBM_F3`):** Refit mở rộng mỗi `26` tuần để sinh phần dư ngoài mẫu đồng nhất phân phối + **cơ chế chặn 1% Validation Guard** (ép `α* = 0` nếu cải thiện trên Val < 1%).
4. **`AutoRouted_Champion` & Split CQR Quantile (`P10/P50/P90`):** Tự động chọn mô hình thắng theo **Validation MAE thấp nhất** (tuyệt đối không chạm Test) và hiệu chuẩn dải tin cậy `P10–P90` bằng Split Conformalized Quantile Regression (Romano et al., 2019).

In [1]:
import json
import time
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")

from pathlib import Path
import os

def resolve_art_dir() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "BTL_BI").is_dir():
            d = p / "BTL_BI" / "artifacts" / "forecasting"
            d.mkdir(parents=True, exist_ok=True)
            return d
        if p.name == "BTL_BI":
            d = p / "artifacts" / "forecasting"
            d.mkdir(parents=True, exist_ok=True)
            return d
    d = cwd / "artifacts" / "forecasting"
    d.mkdir(parents=True, exist_ok=True)
    return d

ART_DIR = resolve_art_dir()
print("Artifacts dir:", ART_DIR)

df_feat = pd.read_parquet(ART_DIR / "feature_store_weekly.parquet")
df_daily = pd.read_parquet(ART_DIR / "feature_store_daily.parquet")
catalog = json.loads((ART_DIR / "feature_catalog.json").read_text(encoding="utf-8"))

f1_cols = catalog["f1_cols"]
f2_cols = catalog["f2_cols"]
f3_compact = catalog["f3_compact_cols"]
f3_lag = catalog["f3_lagged_only_cols"]

f_1 = f1_cols
f_12 = f1_cols + f2_cols
f_123 = f1_cols + f2_cols + f3_compact
f_123_lag = f1_cols + f2_cols + f3_lag

d_12 = catalog["daily_f1_f2_cols"]
d_123 = catalog["daily_all_feature_cols"]

WEEKLY_FOLDS = [
    ("holdout_52w", 208, 250, 302),
    ("roll_f1", 208, 250, 263),
    ("roll_f2", 221, 263, 276),
    ("roll_f3", 234, 276, 289),
    ("roll_f4", 247, 289, 302),
]

assert (df_feat.groupby("atc_code")["week_idx"].count() == 302).all()
assert (df_daily.groupby("atc_code")["day_idx"].count() == 2106).all()
print(f"Weekly: {df_feat.shape} (F1={len(f_1)}, F12={len(f_12)}, F123_Compact={len(f_123)}, LagOnly={len(f_123_lag)})")
print(f"Daily:  {df_daily.shape} (D12={len(d_12)}, D123={len(d_123)})")

Artifacts dir: D:\DOWNLOAD\01_Workspace\DACN\BTL_BI\artifacts\forecasting
Weekly: (2416, 48) (F1=14, F12=22, F123_Compact=29, LagOnly=30)
Daily:  (16848, 46) (D12=18, D123=31)


In [2]:
# Các hàm fit_predict chuẩn hóa (dự báo qty_rate, nhân active_days ra qty; zero leakage)

def make_pred_df(g: pd.DataFrame, val_end: int, te_end: int, fold_id: str, model_name: str, feature_set: str, y_pred: np.ndarray) -> pd.DataFrame:
    return pd.DataFrame({
        "fold_id": fold_id,
        "week_end": g["local_date"].iloc[val_end:te_end].to_numpy(),
        "week_idx": g["week_idx"].iloc[val_end:te_end].to_numpy(),
        "atc_code": g["atc_code"].iloc[0],
        "model_name": model_name,
        "feature_set": feature_set,
        "y_true": g["qty"].iloc[val_end:te_end].to_numpy(),
        "y_pred": np.clip(np.asarray(y_pred, dtype=float), 0.0, None),
    })


def run_seasonal_naive(g: pd.DataFrame, val_end: int, te_end: int, fold_id: str) -> pd.DataFrame:
    y = g["qty"].to_numpy()
    return make_pred_df(g, val_end, te_end, fold_id, "Seasonal_Naive", "lag52_raw_qty", y[val_end - 52 : te_end - 52])


def run_seasonal_naive_weighted(g: pd.DataFrame, val_end: int, te_end: int, fold_id: str) -> pd.DataFrame:
    y = g["qty"].to_numpy()
    pred = (
        0.4 * y[val_end - 52 : te_end - 52]
        + 0.3 * y[val_end - 104 : te_end - 104]
        + 0.2 * y[val_end - 156 : te_end - 156]
        + 0.1 * y[val_end - 208 : te_end - 208]
    )
    return make_pred_df(g, val_end, te_end, fold_id, "Seasonal_Naive_Weighted", "Paper2024_[0.4,0.3,0.2,0.1]", pred)


def select_arima_order(y_seg: np.ndarray, exog_seg: np.ndarray) -> tuple[int, int, int]:
    best_aic, best_order = np.inf, (1, 0, 0)
    for order in [(1, 0, 0), (1, 0, 1), (2, 0, 1), (0, 1, 1), (1, 1, 1)]:
        try:
            fit = ARIMA(y_seg, exog=exog_seg, order=order).fit()
            if np.isfinite(fit.aic) and fit.aic < best_aic:
                best_aic, best_order = float(fit.aic), order
        except Exception:
            continue
    return best_order


def roll_arima_segment(y_rate: np.ndarray, exog: np.ndarray, fit_end: int, roll_end: int, order: tuple[int, int, int]) -> np.ndarray:
    try:
        res = ARIMA(y_rate[:fit_end], exog=exog[:fit_end], order=order).fit()
    except Exception:
        res = ARIMA(y_rate[:fit_end], exog=exog[:fit_end], order=(1, 0, 0)).fit()
    preds = []
    for t in range(fit_end, roll_end):
        fc = np.asarray(res.forecast(steps=1, exog=exog[t : t + 1]))
        preds.append(float(np.clip(fc[0], 0.0, None)))
        res = res.append([y_rate[t]], exog=exog[t : t + 1], refit=False)
    return np.asarray(preds, dtype=float)


def run_harmonic_arima_and_oof(g: pd.DataFrame, tr_end: int, val_end: int, te_end: int, fold_id: str):
    y_rate = g["qty_rate"].to_numpy()
    y_qty = g["qty"].to_numpy()
    ad = g["active_days"].to_numpy()
    exog = g[["week_sin52", "week_cos52", "week_sin26", "week_cos26"]].to_numpy()

    # Expanding-Block 26W OOF SARIMA qua [104:val_end) -> giữ phần dư không trôi & đồng nhất Test
    oof_start = 104
    oof_blocks = []
    for b_start in range(oof_start, val_end, 26):
        b_end = min(b_start + 26, val_end)
        ord_b = select_arima_order(y_rate[:b_start], exog[:b_start])
        oof_blocks.append(roll_arima_segment(y_rate, exog, b_start, b_end, ord_b))
    y_sar_oof_rate = np.concatenate(oof_blocks)
    y_sar_val_rate = y_sar_oof_rate[tr_end - oof_start :]
    val_mae = float(np.mean(np.abs(y_qty[tr_end:val_end] - np.clip(y_sar_val_rate * ad[tr_end:val_end], 0.0, None))))

    # Test [val_end:te_end)
    best_order_te = select_arima_order(y_rate[:val_end], exog[:val_end])
    y_sar_te_rate = roll_arima_segment(y_rate, exog, val_end, te_end, best_order_te)
    y_sar_te_qty = np.clip(y_sar_te_rate * ad[val_end:te_end], 0.0, None)

    df_te = make_pred_df(g, val_end, te_end, fold_id, "Harmonic_ARIMA", f"Rate_Fourier_K2_{best_order_te}", y_sar_te_qty)
    return df_te, val_mae, y_sar_oof_rate, y_sar_te_rate, y_sar_te_qty


def croston_sba_roll(y_r: np.ndarray, alpha: float, start_idx: int, end_idx: int) -> np.ndarray:
    init_len = min(12, start_idx)
    nz_vals = y_r[:init_len][y_r[:init_len] > 0]
    if len(nz_vals) > 0:
        z = float(np.mean(nz_vals))
    else:
        y_pre = y_r[:start_idx]
        z = float(np.mean(y_pre[y_pre > 0])) if np.any(y_pre > 0) else 0.1
    nz_pos = np.where(y_r[:init_len] > 0)[0]
    p = float(np.mean(np.diff(nz_pos))) if len(nz_pos) > 1 else 2.0
    q = 1
    preds = []
    for t in range(init_len, end_idx):
        if t >= start_idx:
            preds.append(max(0.0, (1.0 - alpha / 2.0) * (z / max(p, 1.0))))
        if y_r[t] > 0:
            z = alpha * y_r[t] + (1.0 - alpha) * z
            p = alpha * q + (1.0 - alpha) * p
            q = 1
        else:
            q += 1
    return np.asarray(preds, dtype=float)


def run_croston_sba(g: pd.DataFrame, tr_end: int, val_end: int, te_end: int, fold_id: str):
    y_r = g["qty_rate"].to_numpy()
    y_qty = g["qty"].to_numpy()
    ad = g["active_days"].to_numpy()
    best_alpha, best_mae = 0.1, np.inf
    for alpha in [0.05, 0.1, 0.15, 0.2, 0.3]:
        p_va_qty = np.clip(croston_sba_roll(y_r, alpha, tr_end, val_end) * ad[tr_end:val_end], 0.0, None)
        mae = float(np.mean(np.abs(y_qty[tr_end:val_end] - p_va_qty)))
        if mae < best_mae:
            best_mae, best_alpha = mae, alpha
    y_te_qty = np.clip(croston_sba_roll(y_r, best_alpha, val_end, te_end) * ad[val_end:te_end], 0.0, None)
    return make_pred_df(g, val_end, te_end, fold_id, "Croston_SBA", f"Rate_alpha={best_alpha}", y_te_qty), best_mae, y_te_qty


def run_ridge(g: pd.DataFrame, feature_cols: list[str], tr_end: int, val_end: int, te_end: int, fold_id: str):
    warmup = 52
    tr_m = (g["week_idx"] >= warmup) & (g["week_idx"] < tr_end)
    va_m = (g["week_idx"] >= tr_end) & (g["week_idx"] < val_end)
    tv_m = (g["week_idx"] >= warmup) & (g["week_idx"] < val_end)
    te_m = (g["week_idx"] >= val_end) & (g["week_idx"] < te_end)

    pipe = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 4, 25)))
    pipe.fit(g.loc[tr_m, feature_cols], g.loc[tr_m, "qty_rate"])
    p_va_qty = np.clip(pipe.predict(g.loc[va_m, feature_cols]) * g.loc[va_m, "active_days"].to_numpy(), 0.0, None)
    val_mae = float(np.mean(np.abs(g.loc[va_m, "qty"].to_numpy() - p_va_qty)))

    pipe_tv = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 4, 25)))
    pipe_tv.fit(g.loc[tv_m, feature_cols], g.loc[tv_m, "qty_rate"])
    alpha_chosen = float(pipe_tv.named_steps["ridgecv"].alpha_)
    y_te_qty = np.clip(pipe_tv.predict(g.loc[te_m, feature_cols]) * g.loc[te_m, "active_days"].to_numpy(), 0.0, None)
    return make_pred_df(g, val_end, te_end, fold_id, "Ridge_F1F2F3", f"Rate_alpha={alpha_chosen:.1f}_k{len(feature_cols)}", y_te_qty), val_mae, y_te_qty


def run_lgbm(
    g: pd.DataFrame, feature_cols: list[str], tr_end: int, val_end: int, te_end: int, fold_id: str, model_name: str, atc: str, return_shap: bool = False
):
    warmup = 52
    tr_m = (g["week_idx"] >= warmup) & (g["week_idx"] < tr_end)
    va_m = (g["week_idx"] >= tr_end) & (g["week_idx"] < val_end)
    tv_m = (g["week_idx"] >= warmup) & (g["week_idx"] < val_end)
    te_m = (g["week_idx"] >= val_end) & (g["week_idx"] < te_end)

    X_tr, yr_tr = g.loc[tr_m, feature_cols], g.loc[tr_m, "qty_rate"]
    X_va, yr_va, y_va_q, ad_va = g.loc[va_m, feature_cols], g.loc[va_m, "qty_rate"], g.loc[va_m, "qty"].to_numpy(), g.loc[va_m, "active_days"].to_numpy()
    X_tv, yr_tv = g.loc[tv_m, feature_cols], g.loc[tv_m, "qty_rate"]
    X_te, y_te_q, ad_te = g.loc[te_m, feature_cols], g.loc[te_m, "qty"].to_numpy(), g.loc[te_m, "active_days"].to_numpy()

    if atc == "N05C":
        obj_params = {"objective": "tweedie", "tweedie_variance_power": 1.2, "reg_alpha": 0.2, "reg_lambda": 1.0}
    else:
        obj_params = {"objective": "regression", "reg_alpha": 0.5, "reg_lambda": 2.0}

    best_loss, best_cfg, best_iters = np.inf, {"num_leaves": 7, "learning_rate": 0.03, "min_child_samples": 8}, 80
    for leaves in [7, 15]:
        for lr in [0.03, 0.05]:
            for mcs in [8, 15]:
                m = lgb.LGBMRegressor(
                    n_estimators=500, num_leaves=leaves, learning_rate=lr, min_child_samples=mcs,
                    subsample=0.9, colsample_bytree=0.85, random_state=42, verbosity=-1, **obj_params,
                )
                m.fit(X_tr, yr_tr, eval_set=[(X_va, yr_va)], callbacks=[lgb.early_stopping(30, verbose=False)])
                p_va_qty = np.clip(m.predict(X_va) * ad_va, 0.0, None)
                mae = float(np.mean(np.abs(y_va_q - p_va_qty)))
                if mae < best_loss:
                    best_loss = mae
                    best_cfg = {"num_leaves": leaves, "learning_rate": lr, "min_child_samples": mcs}
                    best_iters = max(20, int(m.best_iteration_ or 80))

    fm = lgb.LGBMRegressor(n_estimators=best_iters, subsample=0.9, colsample_bytree=0.85, random_state=42, verbosity=-1, **best_cfg, **obj_params)
    fm.fit(X_tv, yr_tv)
    y_te_qty = np.clip(fm.predict(X_te) * ad_te, 0.0, None)
    df_out = make_pred_df(g, val_end, te_end, fold_id, model_name, f"Rate_{obj_params['objective']}_n{best_iters}_k{len(feature_cols)}", y_te_qty)

    df_shap = None
    if return_shap:
        contrib = fm.booster_.predict(X_te, pred_contrib=True)
        # Scale L2 rate SHAP contributions by known active_days so unit sum matches qty
        scale = np.ones((len(ad_te), 1)) if atc == "N05C" else ad_te[:, None]
        df_shap = pd.DataFrame(contrib[:, :-1] * scale, columns=feature_cols)
        df_shap.insert(0, "atc_code", atc)
        df_shap.insert(1, "week_idx", g.loc[te_m, "week_idx"].to_numpy())
        df_shap.insert(2, "week_end", g.loc[te_m, "local_date"].to_numpy())
        df_shap.insert(3, "y_true", y_te_q)
        df_shap.insert(4, "y_pred", y_te_qty)
        df_shap["base_value"] = (contrib[:, -1:] * scale).ravel()
        df_shap["shap_space"] = "log_margin" if atc == "N05C" else "raw_units"

    return df_out, best_loss, y_te_qty, df_shap


def run_anchor_res_lgbm(g: pd.DataFrame, feature_cols: list[str], tr_end: int, val_end: int, te_end: int, fold_id: str):
    warmup = 52
    tr_m = (g["week_idx"] >= warmup) & (g["week_idx"] < tr_end)
    va_m = (g["week_idx"] >= tr_end) & (g["week_idx"] < val_end)
    tv_m = (g["week_idx"] >= warmup) & (g["week_idx"] < val_end)
    te_m = (g["week_idx"] >= val_end) & (g["week_idx"] < te_end)

    res_target = g["qty_rate"] - g["ewma_4"]
    X_tr, r_tr = g.loc[tr_m, feature_cols], res_target.loc[tr_m]
    X_va, r_va, ew_va, y_va_q, ad_va = g.loc[va_m, feature_cols], res_target.loc[va_m], g.loc[va_m, "ewma_4"].to_numpy(), g.loc[va_m, "qty"].to_numpy(), g.loc[va_m, "active_days"].to_numpy()
    X_tv, r_tv = g.loc[tv_m, feature_cols], res_target.loc[tv_m]
    X_te, ew_te, ad_te = g.loc[te_m, feature_cols], g.loc[te_m, "ewma_4"].to_numpy(), g.loc[te_m, "active_days"].to_numpy()

    best_loss, best_cfg, best_iters = np.inf, {"learning_rate": 0.03, "min_child_samples": 10}, 50
    for lr in [0.02, 0.04]:
        for mcs in [10, 16]:
            m = lgb.LGBMRegressor(
                objective="regression", n_estimators=400, max_depth=3, num_leaves=7, learning_rate=lr,
                min_child_samples=mcs, reg_alpha=1.0, reg_lambda=5.0, subsample=0.9, colsample_bytree=0.8, random_state=42, verbosity=-1,
            )
            m.fit(X_tr, r_tr, eval_set=[(X_va, r_va)], callbacks=[lgb.early_stopping(30, verbose=False)])
            p_va_qty = np.clip((ew_va + m.predict(X_va)) * ad_va, 0.0, None)
            mae = float(np.mean(np.abs(y_va_q - p_va_qty)))
            if mae < best_loss:
                best_loss = mae
                best_cfg = {"learning_rate": lr, "min_child_samples": mcs}
                best_iters = max(15, int(m.best_iteration_ or 50))

    fm = lgb.LGBMRegressor(
        objective="regression", n_estimators=best_iters, max_depth=3, num_leaves=7, reg_alpha=1.0, reg_lambda=5.0,
        subsample=0.9, colsample_bytree=0.8, random_state=42, verbosity=-1, **best_cfg,
    )
    fm.fit(X_tv, r_tv)
    y_te_qty = np.clip((ew_te + fm.predict(X_te)) * ad_te, 0.0, None)
    return make_pred_df(g, val_end, te_end, fold_id, "LightGBM_AnchorRes_F3", f"Rate_EWMA4Res_n{best_iters}_k{len(feature_cols)}", y_te_qty), best_loss, y_te_qty


def run_hybrid_oof(
    g: pd.DataFrame, feature_cols: list[str], tr_end: int, val_end: int, te_end: int, fold_id: str,
    y_sar_oof_rate: np.ndarray, y_sar_te_rate: np.ndarray,
):
    oof_start = 104
    y_rate = g["qty_rate"].to_numpy()
    y_qty = g["qty"].to_numpy()
    ad = g["active_days"].to_numpy()

    n_tr_oof = tr_end - oof_start
    y_sar_val_rate = y_sar_oof_rate[n_tr_oof:]
    r_tr_oof = y_rate[oof_start:tr_end] - y_sar_oof_rate[:n_tr_oof]
    r_va_oof = y_rate[tr_end:val_end] - y_sar_val_rate
    r_tv_oof = y_rate[oof_start:val_end] - y_sar_oof_rate

    X_tr = g[feature_cols].iloc[oof_start:tr_end]
    X_va = g[feature_cols].iloc[tr_end:val_end]
    X_tv = g[feature_cols].iloc[oof_start:val_end]
    X_te = g[feature_cols].iloc[val_end:te_end]

    res_lgb = lgb.LGBMRegressor(
        objective="regression", n_estimators=400, max_depth=3, num_leaves=7, min_child_samples=12,
        reg_alpha=1.0, reg_lambda=5.0, learning_rate=0.03, subsample=0.9, colsample_bytree=0.8, random_state=42, verbosity=-1,
    )
    res_lgb.fit(X_tr, r_tr_oof, eval_set=[(X_va, r_va_oof)], callbacks=[lgb.early_stopping(30, verbose=False)])
    r_hat_val = res_lgb.predict(X_va)

    # 1% Validation Guard: chi mo alpha* > 0 neu cai thien >= 1.0% MAE tren Validation
    ad_va, y_va_q = ad[tr_end:val_end], y_qty[tr_end:val_end]
    mae_alpha0 = float(np.mean(np.abs(y_va_q - np.clip(y_sar_val_rate * ad_va, 0.0, None))))
    best_alpha, best_mae = 0.0, mae_alpha0
    for alpha in np.linspace(0.05, 1.0, 20):
        cand = np.clip((y_sar_val_rate + alpha * r_hat_val) * ad_va, 0.0, None)
        mae = float(np.mean(np.abs(y_va_q - cand)))
        if mae < best_mae and mae < mae_alpha0 * 0.99:
            best_mae, best_alpha = mae, float(round(alpha, 2))

    best_iters = max(15, int(res_lgb.best_iteration_ or 35))
    if best_alpha == 0.0:
        y_hyb = np.clip(y_sar_te_rate * ad[val_end:te_end], 0.0, None)
    else:
        final_res = lgb.LGBMRegressor(
            objective="regression", n_estimators=best_iters, max_depth=3, num_leaves=7, min_child_samples=12,
            reg_alpha=1.0, reg_lambda=5.0, learning_rate=0.03, subsample=0.9, colsample_bytree=0.8, random_state=42, verbosity=-1,
        )
        final_res.fit(X_tv, r_tv_oof)
        r_hat_te = final_res.predict(X_te)
        y_hyb = np.clip((y_sar_te_rate + best_alpha * r_hat_te) * ad[val_end:te_end], 0.0, None)

    return make_pred_df(g, val_end, te_end, fold_id, "Hybrid_SARIMA_LGBM_F3", f"Rate_Exp26OOF_alpha={best_alpha:.2f}_n{best_iters}", y_hyb), best_mae, y_hyb, best_alpha

In [3]:
# 1. Chạy toàn bộ 5 Folds Tuần + AutoRouted_Champion + TreeSHAP
t0 = time.perf_counter()
preds_all, shap_list, hybrid_alphas, routed_choices = [], [], [], []

for fold_id, tr_end, val_end, te_end in WEEKLY_FOLDS:
    t_f = time.perf_counter()
    for atc, g in df_feat.groupby("atc_code"):
        g = g.sort_values("week_idx").reset_index(drop=True)
        preds_all.append(run_seasonal_naive(g, val_end, te_end, fold_id))
        preds_all.append(run_seasonal_naive_weighted(g, val_end, te_end, fold_id))

        p_sar, mae_sar, y_sar_oof_r, y_sar_te_r, y_sar_te_q = run_harmonic_arima_and_oof(g, tr_end, val_end, te_end, fold_id)
        p_cro, mae_cro, y_cro_te_q = run_croston_sba(g, tr_end, val_end, te_end, fold_id)
        p_rdg, mae_rdg, y_rdg_te_q = run_ridge(g, f_123, tr_end, val_end, te_end, fold_id)
        p_f1, mae_f1, y_f1_te_q, _ = run_lgbm(g, f_1, tr_end, val_end, te_end, fold_id, "LightGBM_F1", atc)
        p_f12, mae_f12, y_f12_te_q, _ = run_lgbm(g, f_12, tr_end, val_end, te_end, fold_id, "LightGBM_F1F2", atc)
        p_f123, mae_f123, y_f123_te_q, df_sh = run_lgbm(g, f_123, tr_end, val_end, te_end, fold_id, "LightGBM_F1F2F3", atc, return_shap=(fold_id == "holdout_52w"))
        p_anc, mae_anc, y_anc_te_q = run_anchor_res_lgbm(g, f_123, tr_end, val_end, te_end, fold_id)
        p_hyb, mae_hyb, y_hyb_te_q, a_star = run_hybrid_oof(g, f_123, tr_end, val_end, te_end, fold_id, y_sar_oof_r, y_sar_te_r)

        preds_all.extend([p_sar, p_cro, p_rdg, p_f1, p_f12, p_f123, p_anc, p_hyb])
        if df_sh is not None:
            shap_list.append(df_sh)
        hybrid_alphas.append({"fold_id": fold_id, "atc_code": atc, "alpha_star": a_star})

        if fold_id == "holdout_52w":
            p_lag, _, _, _ = run_lgbm(g, f_123_lag, tr_end, val_end, te_end, fold_id, "LightGBM_F1F2F3_LagOnly", atc)
            preds_all.append(p_lag)

        # Zero-Leakage Auto-Router (chọn mô hình theo Validation MAE thấp nhất trên [tr_end:val_end))
        candidates = [
            ("Harmonic_ARIMA", mae_sar, y_sar_te_q),
            ("Croston_SBA", mae_cro, y_cro_te_q),
            ("Ridge_F1F2F3", mae_rdg, y_rdg_te_q),
            ("LightGBM_F1", mae_f1, y_f1_te_q),
            ("LightGBM_F1F2", mae_f12, y_f12_te_q),
            ("LightGBM_F1F2F3", mae_f123, y_f123_te_q),
            ("LightGBM_AnchorRes_F3", mae_anc, y_anc_te_q),
            ("Hybrid_SARIMA_LGBM_F3", mae_hyb, y_hyb_te_q),
        ]
        best_cand, best_val_mae, best_te_q = min(candidates, key=lambda x: x[1])
        preds_all.append(make_pred_df(g, val_end, te_end, fold_id, "AutoRouted_Champion", f"selected={best_cand}|valMAE={best_val_mae:.2f}", best_te_q))
        routed_choices.append({"fold_id": fold_id, "atc_code": atc, "selected_model": best_cand, "val_mae": round(best_val_mae, 2)})

    print(f"Finished fold {fold_id} in {time.perf_counter() - t_f:.2f}s")

df_pred_weekly = pd.concat(preds_all, ignore_index=True)
df_holdout = df_pred_weekly[df_pred_weekly["fold_id"] == "holdout_52w"].copy()
df_rolling = df_pred_weekly[df_pred_weekly["fold_id"] != "holdout_52w"].copy()
df_shap = pd.concat(shap_list, ignore_index=True)

assert len(df_holdout) == 8 * 12 * 52, f"Expected 4992 holdout rows, got {len(df_holdout)}"
assert len(df_rolling) == 4 * 8 * 11 * 13, f"Expected 4576 rolling rows, got {len(df_rolling)}"
assert df_pred_weekly["y_pred"].isna().sum() == 0 and (df_pred_weekly["y_pred"] >= 0).all()
assert df_shap.isna().sum().sum() == 0

df_holdout.to_parquet(ART_DIR / "predictions_holdout_52w.parquet", index=False)
df_rolling.to_parquet(ART_DIR / "predictions_rolling_4folds.parquet", index=False)
df_shap.to_parquet(ART_DIR / "shap_values_52w.parquet", index=False)
print("Holdout Hybrid alpha*:", [r for r in hybrid_alphas if r["fold_id"] == "holdout_52w"])
print("Holdout AutoRouted selections:", [r for r in routed_choices if r["fold_id"] == "holdout_52w"])

Finished fold holdout_52w in 29.37s


Finished fold roll_f1 in 27.52s


Finished fold roll_f2 in 32.26s


Finished fold roll_f3 in 33.11s


Finished fold roll_f4 in 36.71s
Holdout Hybrid alpha*: [{'fold_id': 'holdout_52w', 'atc_code': 'M01AB', 'alpha_star': 1.0}, {'fold_id': 'holdout_52w', 'atc_code': 'M01AE', 'alpha_star': 0.0}, {'fold_id': 'holdout_52w', 'atc_code': 'N02BA', 'alpha_star': 0.0}, {'fold_id': 'holdout_52w', 'atc_code': 'N02BE', 'alpha_star': 0.0}, {'fold_id': 'holdout_52w', 'atc_code': 'N05B', 'alpha_star': 0.0}, {'fold_id': 'holdout_52w', 'atc_code': 'N05C', 'alpha_star': 0.0}, {'fold_id': 'holdout_52w', 'atc_code': 'R03', 'alpha_star': 1.0}, {'fold_id': 'holdout_52w', 'atc_code': 'R06', 'alpha_star': 0.85}]
Holdout AutoRouted selections: [{'fold_id': 'holdout_52w', 'atc_code': 'M01AB', 'selected_model': 'Croston_SBA', 'val_mae': 5.72}, {'fold_id': 'holdout_52w', 'atc_code': 'M01AE', 'selected_model': 'Ridge_F1F2F3', 'val_mae': 4.78}, {'fold_id': 'holdout_52w', 'atc_code': 'N02BA', 'selected_model': 'Harmonic_ARIMA', 'val_mae': 3.72}, {'fold_id': 'holdout_52w', 'atc_code': 'N02BE', 'selected_model': 'Light

In [4]:
# 2. Split Conformalized Global LightGBM Quantile (P10 / P50 / P90) — Romano et al. (2019)
t_q = time.perf_counter()

def pinball(y: np.ndarray, yhat: np.ndarray, q: float) -> float:
    e = y - yhat
    return float(np.mean(np.maximum(q * e, (q - 1.0) * e)))

g_df = df_feat.sort_values(["atc_code", "week_idx"]).copy()
g_df["atc_code_cat"] = g_df["atc_code"].astype("category")
q_cols = ["atc_code_cat"] + f_123

tr_m = (g_df["week_idx"] >= 52) & (g_df["week_idx"] < 208)
va_m = (g_df["week_idx"] >= 208) & (g_df["week_idx"] < 250)
tv_m = (g_df["week_idx"] >= 52) & (g_df["week_idx"] < 250)
te_m = (g_df["week_idx"] >= 250) & (g_df["week_idx"] < 302)

X_tr, yr_tr = g_df.loc[tr_m, q_cols], g_df.loc[tr_m, "qty_rate"]
X_va, y_va_q, ad_va = g_df.loc[va_m, q_cols], g_df.loc[va_m, "qty"].to_numpy(), g_df.loc[va_m, "active_days"].to_numpy()
X_tv, yr_tv = g_df.loc[tv_m, q_cols], g_df.loc[tv_m, "qty_rate"]
X_te, y_te_q, ad_te = g_df.loc[te_m, q_cols], g_df.loc[te_m, "qty"].to_numpy(), g_df.loc[te_m, "active_days"].to_numpy()

q_va_qty, q_te_raw = {}, {}
for q_lvl, tag in [(0.10, "q10"), (0.50, "q50"), (0.90, "q90")]:
    best_pb, best_cfg, best_iters, best_m = np.inf, {"num_leaves": 15, "learning_rate": 0.03}, 100, None
    for leaves in [15, 31]:
        for lr in [0.03, 0.05]:
            m = lgb.LGBMRegressor(
                objective="quantile", alpha=q_lvl, n_estimators=600, num_leaves=leaves, learning_rate=lr,
                min_child_samples=15, subsample=0.9, colsample_bytree=0.9, random_state=42, verbosity=-1,
            )
            m.fit(X_tr, yr_tr, eval_set=[(X_va, g_df.loc[va_m, "qty_rate"])], callbacks=[lgb.early_stopping(35, verbose=False)])
            p_va_q = np.clip(m.predict(X_va) * ad_va, 0.0, None)
            pb = pinball(y_va_q, p_va_q, q_lvl)
            if pb < best_pb:
                best_pb, best_cfg, best_iters, best_m = pb, {"num_leaves": leaves, "learning_rate": lr}, max(25, int(m.best_iteration_ or 100)), m
    q_va_qty[tag] = np.clip(best_m.predict(X_va) * ad_va, 0.0, None)

    fm = lgb.LGBMRegressor(
        objective="quantile", alpha=q_lvl, n_estimators=best_iters, min_child_samples=15,
        subsample=0.9, colsample_bytree=0.9, random_state=42, verbosity=-1, **best_cfg,
    )
    fm.fit(X_tv, yr_tv)
    q_te_raw[tag] = np.clip(fm.predict(X_te) * ad_te, 0.0, None)

raw_mat = np.sort(np.column_stack([q_te_raw["q10"], q_te_raw["q50"], q_te_raw["q90"]]), axis=1)

# Per-ATC Split Conformal Quantile calibration trên Validation W209..W250 (mục tiêu coverage >= 80%)
va_atc = g_df.loc[va_m, "atc_code"].to_numpy()
te_atc = g_df.loc[te_m, "atc_code"].to_numpy()
qhat_map = {}
for atc in sorted(np.unique(va_atc)):
    mask = va_atc == atc
    n_va = int(mask.sum())
    scores = np.maximum(q_va_qty["q10"][mask] - y_va_q[mask], y_va_q[mask] - q_va_qty["q90"][mask])
    lvl = min(1.0, float(np.ceil(0.80 * (n_va + 1))) / n_va)
    qhat_map[atc] = float(np.quantile(scores, lvl, method="higher"))

te_qhat = np.array([qhat_map[a] for a in te_atc])
cqr_mat = np.sort(np.column_stack([
    np.clip(raw_mat[:, 0] - te_qhat, 0.0, None),
    np.clip(raw_mat[:, 1], 0.0, None),
    np.clip(raw_mat[:, 2] + te_qhat, 0.0, None),
]), axis=1)

df_quant = pd.DataFrame({
    "week_end": g_df.loc[te_m, "local_date"].to_numpy(),
    "week_idx": g_df.loc[te_m, "week_idx"].to_numpy(),
    "atc_code": te_atc,
    "y_true": y_te_q,
    "q10_raw": raw_mat[:, 0],
    "q50_raw": raw_mat[:, 1],
    "q90_raw": raw_mat[:, 2],
    "cqr_qhat": te_qhat,
    "q10": cqr_mat[:, 0],
    "q50": cqr_mat[:, 1],
    "q90": cqr_mat[:, 2],
})
assert (df_quant["q10"] <= df_quant["q50"]).all() and (df_quant["q50"] <= df_quant["q90"]).all()
df_quant.to_parquet(ART_DIR / "predictions_quantiles_52w.parquet", index=False)
print("Split CQR per-ATC q_hat:", {k: round(v, 2) for k, v in qhat_map.items()})

Split CQR per-ATC q_hat: {'M01AB': 1.88, 'M01AE': 0.4, 'N02BA': -0.61, 'N02BE': 22.72, 'N05B': -4.88, 'N05C': 1.25, 'R03': 7.4, 'R06': 1.5}


In [5]:
# 3. Daily Track (Test = 365D: day_idx 1741..2105, Val = 180D: 1561..1740, Train Core = 364..1560)
t_d = time.perf_counter()
daily_preds = []
D_WARMUP, D_TR_END, D_VA_END, D_TE_END = 364, 1561, 1741, 2106

for atc, g in df_daily.groupby("atc_code"):
    g = g.sort_values("day_idx").reset_index(drop=True)
    y = g["qty"].to_numpy()
    te_dates = g["local_date"].iloc[D_VA_END:D_TE_END].to_numpy()
    te_idx = g["day_idx"].iloc[D_VA_END:D_TE_END].to_numpy()
    y_te = y[D_VA_END:D_TE_END]

    daily_preds.append(pd.DataFrame({
        "local_date": te_dates, "day_idx": te_idx, "atc_code": atc,
        "model_name": "Seasonal_Naive_7D", "y_true": y_te, "y_pred": np.clip(y[D_VA_END - 7 : D_TE_END - 7], 0.0, None),
    }))

    tr_m = (g["day_idx"] >= D_WARMUP) & (g["day_idx"] < D_TR_END)
    va_m = (g["day_idx"] >= D_TR_END) & (g["day_idx"] < D_VA_END)
    tv_m = (g["day_idx"] >= D_WARMUP) & (g["day_idx"] < D_VA_END)
    te_m = (g["day_idx"] >= D_VA_END) & (g["day_idx"] < D_TE_END)

    obj_p = {"objective": "tweedie", "tweedie_variance_power": 1.2} if atc == "N05C" else {"objective": "regression", "reg_alpha": 0.3, "reg_lambda": 1.5}
    for m_name, cols in [("LightGBM_Daily_F1F2", d_12), ("LightGBM_Daily_F1F2F3", d_123)]:
        best_mae, best_cfg, best_iters = np.inf, {"num_leaves": 15, "learning_rate": 0.03, "min_child_samples": 14}, 100
        for leaves in [15, 31]:
            for lr in [0.03, 0.05]:
                for mcs in [14, 28]:
                    m = lgb.LGBMRegressor(n_estimators=500, num_leaves=leaves, learning_rate=lr, min_child_samples=mcs, subsample=0.9, colsample_bytree=0.9, random_state=42, verbosity=-1, **obj_p)
                    m.fit(g.loc[tr_m, cols], g.loc[tr_m, "qty"], eval_set=[(g.loc[va_m, cols], g.loc[va_m, "qty"])], callbacks=[lgb.early_stopping(30, verbose=False)])
                    mae = float(np.mean(np.abs(g.loc[va_m, "qty"].to_numpy() - np.clip(m.predict(g.loc[va_m, cols]), 0.0, None))))
                    if mae < best_mae:
                        best_mae, best_cfg = mae, {"num_leaves": leaves, "learning_rate": lr, "min_child_samples": mcs}
                        best_iters = max(25, int(m.best_iteration_ or 100))

        fm = lgb.LGBMRegressor(n_estimators=best_iters, subsample=0.9, colsample_bytree=0.9, random_state=42, verbosity=-1, **best_cfg, **obj_p)
        fm.fit(g.loc[tv_m, cols], g.loc[tv_m, "qty"])
        daily_preds.append(pd.DataFrame({
            "local_date": te_dates, "day_idx": te_idx, "atc_code": atc,
            "model_name": m_name, "y_true": y_te, "y_pred": np.clip(fm.predict(g.loc[te_m, cols]), 0.0, None),
        }))

df_daily_pred = pd.concat(daily_preds, ignore_index=True)
assert len(df_daily_pred) == 8 * 3 * 365
assert df_daily_pred["y_pred"].isna().sum() == 0 and (df_daily_pred["y_pred"] >= 0).all()
df_daily_pred.to_parquet(ART_DIR / "predictions_daily_365d.parquet", index=False)

total_elapsed = time.perf_counter() - t0
summary = {
    "total_elapsed_seconds": round(total_elapsed, 2),
    "daily_elapsed_seconds": round(time.perf_counter() - t_d, 2),
    "weekly_folds": WEEKLY_FOLDS,
    "holdout_rows": int(len(df_holdout)),
    "rolling_rows": int(len(df_rolling)),
    "quantile_rows": int(len(df_quant)),
    "daily_rows": int(len(df_daily_pred)),
    "hybrid_alphas": hybrid_alphas,
    "autorouted_choices": routed_choices,
    "cqr_qhat_map": {k: round(v, 3) for k, v in qhat_map.items()},
}
(ART_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
print(f"COMPLETED UNIFIED TRAINING/BACKTEST PIPELINE IN {total_elapsed:.2f}s (vs Paper 2024 54h Grid Search)!")

COMPLETED UNIFIED TRAINING/BACKTEST PIPELINE IN 166.60s (vs Paper 2024 54h Grid Search)!
